# Shallow 100-dimensional Poisson PINN: affine–quadratic hybrid certification

This notebook trains (when explicitly requested) and certifies a shallow
$100$–$300$–$1$ tanh PINN for the same manufactured Poisson problem as the
deep benchmark.  The committed checkpoint is loaded by default.  Training uses
only the PDE residual and sampled Dirichlet boundary loss; the exact solution
is reserved for validation.

The primary certified comparison uses one global cell on
$[-0.1,0.1]^{100}$:

- interval propagation;
- affine-PZ value/one-jet propagation with symbolic integration;
- the affine–quadratic hybrid PZ one-jet, using a certified quadratic
  approximation of $\tanh'$ only on zero-crossing preactivation intervals
  whose relative affine slope is at most $0.01$; the primary hybrid run is
  uncompressed and uses scalar-output reverse-mode Jacobian propagation plus
  direct structured symbolic integration.

All canonical outputs implement schema 1.2 of
`docs/diagnostics_and_metrics_glossary.tex`.

In [1]:
from __future__ import annotations

import math
import random
import sys
from pathlib import Path
from time import perf_counter

import numpy as np
import torch
from torch import nn

repo_root = Path.cwd()
while not (repo_root / 'src' / 'intervalnets').exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from intervalnets import (
    IntervalTensor,
    PZIntegrationCell,
    enable_interval_eval,
    integrate_pz_onejet_squared,
    integrate_pz_value_squared,
    integrate_shallow_hybrid_onejet_squared,
    load_tanh_mlp_checkpoint,
    sequential_value_jacobian_laplacian,
    shallow_scalar_hybrid_onejet_reverse,
)

torch.set_num_threads(1)
torch.set_default_dtype(torch.float64)
enable_interval_eval()

DIM = 100
HALF_WIDTH = 0.1
HIDDEN = (300,)
SEED = 20260804
K1 = 2.5
K2 = 1.75
COS_AMPLITUDE = 0.35
CHECKPOINT = repo_root / 'notebooks' / 'checkpoints' / 'pinn_100d_poisson_shallow_300.pt'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f'torch={torch.__version__}, dtype={torch.get_default_dtype()}, threads={torch.get_num_threads()}')
print(f'checkpoint={CHECKPOINT.relative_to(repo_root)}')

torch=2.8.0+cpu, dtype=torch.float64, threads=1
checkpoint=notebooks/checkpoints/pinn_100d_poisson_shallow_300.pt


## PDE, architecture, and reproducible checkpoint

In [2]:
def dense_directions(dim=DIM):
    a = torch.ones(dim)
    a /= torch.linalg.vector_norm(a)
    b = torch.tensor([1.0 if i % 2 == 0 else -1.0 for i in range(dim)])
    b -= torch.dot(a, b) * a
    b /= torch.linalg.vector_norm(b)
    return a, b


A, B = dense_directions()


def exact_solution(x):
    return (torch.sin(K1 * (x @ A)) + COS_AMPLITUDE * torch.cos(K2 * (x @ B))).unsqueeze(-1)


def exact_gradient(x):
    s = x @ A
    t = x @ B
    return K1 * torch.cos(K1 * s).unsqueeze(-1) * A - COS_AMPLITUDE * K2 * torch.sin(K2 * t).unsqueeze(-1) * B


def forcing(x):
    return (K1**2 * torch.sin(K1 * (x @ A)) + COS_AMPLITUDE * K2**2 * torch.cos(K2 * (x @ B))).unsqueeze(-1)


def make_model():
    layers, previous = [], DIM
    for width in HIDDEN:
        layers.extend([nn.Linear(previous, width), nn.Tanh()])
        previous = width
    layers.append(nn.Linear(previous, 1))
    model = nn.Sequential(*layers)
    for layer in model:
        if isinstance(layer, nn.Linear):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
    return model


def sample_interior(n, generator):
    return (2.0 * torch.rand((n, DIM), generator=generator) - 1.0) * HALF_WIDTH


def sample_boundary(n, generator):
    x = sample_interior(n, generator)
    coordinate = torch.randint(DIM, (n,), generator=generator)
    sign = torch.where(torch.rand(n, generator=generator) < 0.5, -1.0, 1.0)
    x[torch.arange(n), coordinate] = HALF_WIDTH * sign
    return x


def pinn_residual(model, x):
    value, jacobian, laplacian = sequential_value_jacobian_laplacian(model, x)
    return value, jacobian, -laplacian - forcing(x)


model = make_model()
sum(parameter.numel() for parameter in model.parameters()), model

(30601,
 Sequential(
   (0): Linear(in_features=100, out_features=300, bias=True)
   (1): Tanh()
   (2): Linear(in_features=300, out_features=1, bias=True)
 ))

In [3]:
def train_pinn(model, steps=2200, batch_size=512, lr=2e-3):
    generator = torch.Generator().manual_seed(SEED + 1)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=steps, eta_min=1e-4)
    history = []
    model.train()
    for step in range(1, steps + 1):
        interior = sample_interior(batch_size, generator)
        boundary = sample_boundary(batch_size, generator)
        _, _, residual = pinn_residual(model, interior)
        boundary_error = model(boundary) - exact_solution(boundary)
        residual_loss = residual.square().mean()
        boundary_loss = boundary_error.square().mean()
        loss = residual_loss + 20.0 * boundary_loss
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        optimizer.step()
        scheduler.step()
        if step == 1 or step % 100 == 0:
            history.append({
                'step': step,
                'loss': float(loss.detach()),
                'residual_loss': float(residual_loss.detach()),
                'boundary_loss': float(boundary_loss.detach()),
            })
    model.eval()
    return history


RETRAIN = False
if RETRAIN:
    training_start = perf_counter()
    training_history = train_pinn(model)
    training_seconds = perf_counter() - training_start
    checkpoint_payload = {
        'state_dict': model.state_dict(),
        'seed': SEED,
        'architecture': [DIM, *HIDDEN, 1],
        'problem': 'poisson_100d_ridge',
        'domain_half_width': HALF_WIDTH,
        'training': {
            'steps': 2200,
            'batch_size': 512,
            'optimizer': 'Adam',
            'initial_lr': 2e-3,
            'final_lr': 1e-4,
            'boundary_weight': 20.0,
            'seconds': training_seconds,
            'history': training_history,
        },
    }
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    torch.save(checkpoint_payload, CHECKPOINT)
else:
    model = load_tanh_mlp_checkpoint(CHECKPOINT)
    checkpoint_payload = torch.load(CHECKPOINT, map_location='cpu', weights_only=True)
    training_history = []

{'loaded_checkpoint': not RETRAIN,
 'architecture': checkpoint_payload.get('architecture'),
 'training_steps': checkpoint_payload.get('training', {}).get('steps'),
 'training_seconds': checkpoint_payload.get('training', {}).get('seconds'),
 'training_records': training_history[-3:]}


{'loaded_checkpoint': True,
 'architecture': [100, 300, 1],
 'training_steps': 2200,
 'training_seconds': 262.240752271,
 'training_records': []}

## Independent sampled validation

In [4]:
validation_generator = torch.Generator().manual_seed(SEED + 222)
interior = sample_interior(16384, validation_generator)
boundary = sample_boundary(16384, validation_generator)
with torch.no_grad():
    prediction, network_jacobian, residual = pinn_residual(model, interior)
    target = exact_solution(interior)
    target_gradient = exact_gradient(interior)
    boundary_error = model(boundary) - exact_solution(boundary)
    error = prediction - target
    empirical_network_l2 = prediction.square().mean().sqrt()
    empirical_network_w12 = (prediction.square() + network_jacobian.square().sum(dim=(-2, -1), keepdim=True)).mean().sqrt()
    empirical_exact_l2 = target.square().mean().sqrt()
    empirical_exact_w12 = (target.square() + target_gradient.square().sum(dim=-1, keepdim=True)).mean().sqrt()

validation = {
    'solution_RMSE': float(error.square().mean().sqrt()),
    'solution_relative_L2_error': float(error.square().mean().sqrt() / target.square().mean().sqrt()),
    'solution_max_sample_error': float(error.abs().max()),
    'gradient_RMSE': float((network_jacobian.squeeze(1) - target_gradient).square().mean().sqrt()),
    'PDE_residual_RMSE': float(residual.square().mean().sqrt()),
    'boundary_RMSE': float(boundary_error.square().mean().sqrt()),
    'network_normalized_L2_MC': float(empirical_network_l2),
    'network_normalized_W12_MC': float(empirical_network_w12),
    'exact_normalized_L2_MC': float(empirical_exact_l2),
    'exact_normalized_W12_MC': float(empirical_exact_w12),
}
validation
if RETRAIN:
    checkpoint_payload['validation'] = validation
    torch.save(checkpoint_payload, CHECKPOINT)
validation


{'solution_RMSE': 0.002764527838374597,
 'solution_relative_L2_error': 0.007386462596925562,
 'solution_max_sample_error': 0.027334537779656998,
 'gradient_RMSE': 0.007121730317469054,
 'PDE_residual_RMSE': 0.013159160676084783,
 'boundary_RMSE': 0.002782393594360182,
 'network_normalized_L2_MC': 0.374310346352375,
 'network_normalized_W12_MC': 2.502324834012471,
 'exact_normalized_L2_MC': 0.3742695237535306,
 'exact_normalized_W12_MC': 2.503330852539675}

## Domain and symbolic-integration helpers

In [5]:
SQRT_VOLUME = (2.0 * HALF_WIDTH) ** (DIM / 2.0)
VOLUME = SQRT_VOLUME ** 2
DOMAIN = IntervalTensor.from_bounds([-HALF_WIDTH] * DIM, [HALF_WIDTH] * DIM)


def norm_interval(squared):
    return math.sqrt(max(0.0, float(squared.lower))), math.sqrt(max(0.0, float(squared.upper)))


def interval_metrics(bounds, prefix):
    lower, upper = map(float, bounds)
    width = upper - lower
    return {
        f'{prefix}_lower': lower,
        f'{prefix}_upper': upper,
        f'{prefix}_absolute_width': width,
        f'{prefix}_relative_width': width / max(abs(lower), abs(upper)) if max(abs(lower), abs(upper)) > 0.0 else 0.0,
        f'{prefix}_normalized_lower': lower / SQRT_VOLUME,
        f'{prefix}_normalized_upper': upper / SQRT_VOLUME,
        f'{prefix}_normalized_absolute_width': width / SQRT_VOLUME,
    }


def jacobian_width_metrics(enclosure):
    lower = torch.as_tensor(enclosure.lower)
    upper = torch.as_tensor(enclosure.upper)
    widths = upper - lower
    scales = torch.maximum(lower.abs(), upper.abs())
    relative_widths = torch.where(scales > 0.0, widths / scales, 0.0)
    return {
        'J_mean_component_width_before_integration': float(widths.mean()),
        'J_max_component_width_before_integration': float(widths.max()),
        'J_relative_mean_component_width_before_integration': float(relative_widths.mean()),
    }


def benchmark_polynomial(model, strategy='topk', **kwargs):
    cell = PZIntegrationCell.from_affine_box(DOMAIN)
    start = perf_counter()
    traced = model.eval_pz_onejet(
        cell.domain, return_trace=True, reduction_strategy=strategy, **kwargs
    )
    forward_s = perf_counter() - start
    jacobian_metrics = jacobian_width_metrics(traced.final.J.interval_enclosure())
    start = perf_counter()
    l2_integrated_pz = integrate_pz_value_squared(traced.final.Y, cell, output='pz')
    l2_squared = l2_integrated_pz.interval_enclosure()
    l2_integration_s = perf_counter() - start
    start = perf_counter()
    w12_integrated_pz = integrate_pz_onejet_squared(traced.final, cell, output='pz')
    w12_squared = w12_integrated_pz.interval_enclosure()
    w12_integration_s = perf_counter() - start
    return {
        'strategy': strategy,
        **kwargs,
        'forward_s': forward_s,
        'L2_integration_s': l2_integration_s,
        'W12_integration_s': w12_integration_s,
        'total_W12_s': forward_s + w12_integration_s,
        'J_terms': len(traced.final.J.terms),
        'J_degree': max(map(sum, traced.final.J.terms), default=0),
        'noise_count': traced.final.J.num_noise,
        'L2_integrated_PZ_terms': len(l2_integrated_pz.terms),
        'W12_integrated_PZ_terms': len(w12_integrated_pz.terms),
        'W12_integrated_PZ_noise': w12_integrated_pz.num_noise,
        **jacobian_metrics,
        **interval_metrics(norm_interval(l2_squared), 'L2'),
        **interval_metrics(norm_interval(w12_squared), 'W12'),
        'trace': traced.records,
    }


SQRT_VOLUME

1.1258999068426271e-35

## Glossary-conformant medium benchmark

The medium benchmark writes the complete mini-benchmark outputs plus the
per-neuron activation table and the layerwise normalized postactivation-radius
table.  The value activation always uses the certified affine enclosure.  For
the hybrid derivative, the quadratic polynomial coefficient remains in the PZ
core and its certified approximation-error radius $\rho_{0i}^{(1)}$ is kept
as one shared generator per neuron. The primary hybrid run is fully uncompressed:
it uses reverse-mode Jacobian propagation, retains every degree-one and degree-two
domain monomial, and performs direct symbolic integration of the structured square.

In [6]:
import json
import subprocess
from datetime import datetime, timezone

import pandas as pd

from intervalnets import interval_forward

SCHEMA_VERSION = "1.2"
BENCHMARK_LEVEL = "medium"
PROBLEM_ID = "poisson_100d_ridge"
MODEL_ID = "pinn_100d_poisson_shallow_300_seed_20260804"
SPLIT_ID = "single_cell"
OUTPUT_ROOT = repo_root / "notebooks" / "benchmark_outputs" / "pinn_100d_poisson_shallow_300_medium"

METRICS_COLUMNS = [
    "schema_version", "benchmark_level", "problem_id", "model_id", "method_id",
    "run_id", "split_id", "quantity", "metric", "aggregation",
    "derivative_order", "layer", "neuron", "output_index", "input_index_a",
    "input_index_b", "cell_id", "value", "unit", "status",
]
CELL_INTERVAL_COLUMNS = [
    "run_id", "split_id", "cell_id", "cell_weight", "quantity", "output_index",
    "input_index_a", "input_index_b", "lower", "upper", "midpoint", "radius",
    "width", "magnitude", "mignitude", "local_relative_radius",
    "global_normalized_radius", "sign_certified", "status",
    "local_relative_width",
]
NORM_COLUMNS = [
    "run_id", "norm", "squared", "lower", "upper", "width", "relative_width",
    "value_contribution_upper", "gradient_contribution_upper",
    "hessian_contribution_upper", "value_contribution_width",
    "gradient_contribution_width", "hessian_contribution_width", "status",
    "domain_volume", "domain_volume_normalized_lower",
    "domain_volume_normalized_upper", "domain_volume_normalized_width",
]
ACTIVATION_COLUMNS = [
    "run_id", "split_id", "cell_id", "cell_weight", "layer", "neuron",
    "derivative_order", "preactivation_lower", "preactivation_upper",
    "preactivation_midpoint", "preactivation_radius", "preactivation_width",
    "approximation_kind", "approximation_error_radius",
    "approximation_error_diameter", "activation_scale",
    "normalized_approximation_radius", "noise_symbol_id", "shared_noise_group",
    "status",
]
COMPLEXITY_COLUMNS = [
    "run_id", "quantity", "n_alpha", "n_eta", "n_monomials",
    "n_mixed_monomials", "max_degree", "n_coefficients", "status",
]
TIMING_COLUMNS = ["run_id", "stage", "seconds", "status"]
SOUNDNESS_COLUMNS = [
    "run_id", "quantity", "sample_count", "failure_count", "max_violation",
    "invalid_interval_count", "nan_endpoint_count", "infinite_endpoint_count",
    "status",
]


def _tensor(value):
    return torch.as_tensor(value, dtype=torch.get_default_dtype()).detach().cpu()


def _nonnegative_squared_interval(enclosure):
    lower = max(0.0, float(enclosure.lower))
    upper = max(0.0, float(enclosure.upper))
    return lower, upper


def _outward_square(bounds):
    lower, upper = (max(0.0, float(v)) for v in bounds)
    return (
        float(np.nextafter(lower * lower, -np.inf)) if lower else 0.0,
        float(np.nextafter(upper * upper, np.inf)) if upper else 0.0,
    )


def _interval_trace(model, domain):
    current = domain
    records = []
    hidden_layer = 0
    for child in model:
        if isinstance(child, nn.Linear):
            current = interval_forward(child, current, enclosure_mode="box")
        elif isinstance(child, nn.Tanh):
            preactivation = current
            current = interval_forward(child, current, enclosure_mode="box")
            records.append({
                "layer": hidden_layer,
                "preactivation_lower": _tensor(preactivation.lower).reshape(-1),
                "preactivation_upper": _tensor(preactivation.upper).reshape(-1),
                "postactivation_lower": _tensor(current.lower).reshape(-1),
                "postactivation_upper": _tensor(current.upper).reshape(-1),
            })
            hidden_layer += 1
        else:
            current = interval_forward(child, current, enclosure_mode="box")
    return current, records



def _run_medium_methods(model):
    results = {}

    interval_start = perf_counter()
    interval_l2 = model.lpnorm(DOMAIN, p=2.0, method="interval")
    interval_l2_s = perf_counter() - interval_start
    interval_start = perf_counter()
    interval_w12 = model.sobolev_norm(DOMAIN, p=2.0, order=1, method="interval")
    interval_w12_s = perf_counter() - interval_start
    interval_start = perf_counter()
    interval_y, interval_activation = _interval_trace(model, DOMAIN)
    interval_j = model.eval_jacobian(DOMAIN)
    interval_enclosure_s = perf_counter() - interval_start
    results["interval"] = {
        "method_id": "interval",
        "run_id": "pinn100d_shallow300_medium_interval",
        "Y": interval_y,
        "J": interval_j,
        "activation": interval_activation,
        "norms": {
            "L2": (float(interval_l2.lower), float(interval_l2.upper)),
            "L2_sq": _outward_square((interval_l2.lower, interval_l2.upper)),
            "W12": (float(interval_w12.lower), float(interval_w12.upper)),
            "W12_sq": _outward_square((interval_w12.lower, interval_w12.upper)),
        },
        "timings": {
            "L2_total": interval_l2_s,
            "W12_total": interval_w12_s,
            "final_enclosures": interval_enclosure_s,
        },
    }

    cell = PZIntegrationCell.from_affine_box(DOMAIN)
    configurations = {
        "affine_pz_topk96_symbolic": {
            "run_id": "pinn100d_shallow300_medium_affine_pz_topk96_symbolic",
            "derivative_enclosure": "affine",
            "derivative_flatness_threshold": 0.01,
        },
        "hybrid_pz_topk96_B_symbolic": {
            "run_id": "pinn100d_shallow300_medium_hybrid_pz_topk96_B_symbolic",
            "derivative_enclosure": "quadratic_flat",
            "derivative_flatness_threshold": 0.01,
        },
    }
    for method_id, config in configurations.items():
        pz_start = perf_counter()
        traced = model.eval_pz_onejet(
            cell.domain,
            return_trace=True,
            reduction_strategy="topk",
            max_terms=96,
            reduction_variant="B",
            derivative_enclosure=config["derivative_enclosure"],
            derivative_flatness_threshold=config["derivative_flatness_threshold"],
            quadratic_certificate_subdivisions=64,
            quadratic_compression_guard=False,
        )
        pz_forward_s = perf_counter() - pz_start
        pz_l2_start = perf_counter()
        pz_l2_integrated = integrate_pz_value_squared(traced.final.Y, cell, output="pz")
        pz_l2_sq = pz_l2_integrated.interval_enclosure()
        pz_l2_s = perf_counter() - pz_l2_start
        pz_w12_start = perf_counter()
        pz_w12_integrated = integrate_pz_onejet_squared(traced.final, cell, output="pz")
        pz_w12_sq = pz_w12_integrated.interval_enclosure()
        pz_w12_s = perf_counter() - pz_w12_start
        pz_activation = []
        hidden_layer = 0
        for record in traced.records:
            if record.layer_type != "Tanh":
                continue
            postactivation = record.value.interval_enclosure()
            degrees = record.summary["tanh_prime_approximation_degrees"].detach().cpu().numpy()
            pz_activation.append({
                "layer": hidden_layer,
                "preactivation_lower": record.summary["preactivation_lower"].detach().cpu(),
                "preactivation_upper": record.summary["preactivation_upper"].detach().cpu(),
                "postactivation_lower": _tensor(postactivation.lower).reshape(-1),
                "postactivation_upper": _tensor(postactivation.upper).reshape(-1),
                "rho0": record.summary["tanh_approximation_radii"].detach().cpu(),
                "rho1": record.summary["tanh_prime_approximation_radii"].detach().cpu(),
                "kind0": np.full(len(degrees), "affine", dtype=object),
                "kind1": np.where(degrees == 2, "quadratic", "affine"),
                "affine_rho1": record.summary["tanh_prime_affine_radii"].detach().cpu(),
                "quadratic_core_box_radius": record.summary["tanh_prime_polynomial_reduction_radii"].detach().cpu(),
                "relative_slope": record.summary["tanh_prime_relative_slopes"].detach().cpu(),
            })
            hidden_layer += 1
        pz_l2_sq_bounds = _nonnegative_squared_interval(pz_l2_sq)
        pz_w12_sq_bounds = _nonnegative_squared_interval(pz_w12_sq)
        results[method_id] = {
            "method_id": method_id,
            "run_id": config["run_id"],
            "Y": traced.final.Y.interval_enclosure(),
            "J": traced.final.J.interval_enclosure(),
            "Y_pz": traced.final.Y,
            "J_pz": traced.final.J,
            "activation": pz_activation,
            "trace": traced.records,
            "integrated_L2_pz": pz_l2_integrated,
            "integrated_W12_pz": pz_w12_integrated,
            "norms": {
                "L2_sq": pz_l2_sq_bounds,
                "L2": norm_interval(pz_l2_sq),
                "W12_sq": pz_w12_sq_bounds,
                "W12": norm_interval(pz_w12_sq),
            },
            "timings": {
                "onejet_construction": pz_forward_s,
                "L2_symbolic_integration": pz_l2_s,
                "W12_symbolic_integration": pz_w12_s,
                "W12_total": pz_forward_s + pz_w12_s,
            },
        }

    method_id = "hybrid_pz_uncompressed_reverse_symbolic"
    reverse_start = perf_counter()
    reverse = shallow_scalar_hybrid_onejet_reverse(
        model,
        cell.domain,
        derivative_flatness_threshold=0.01,
        quadratic_certificate_subdivisions=64,
    )
    reverse_forward_s = perf_counter() - reverse_start
    pz_l2_start = perf_counter()
    pz_l2_integrated = integrate_pz_value_squared(reverse.final.Y, cell, output="pz")
    pz_l2_sq = pz_l2_integrated.interval_enclosure()
    pz_l2_s = perf_counter() - pz_l2_start
    pz_w12_start = perf_counter()
    pz_w12_integrated = integrate_shallow_hybrid_onejet_squared(reverse, cell, output="pz")
    pz_w12_sq = pz_w12_integrated.interval_enclosure()
    pz_w12_s = perf_counter() - pz_w12_start
    compressed_activation = results["hybrid_pz_topk96_B_symbolic"]["activation"][0]
    degrees = reverse.derivative_degrees.detach().cpu().numpy()
    reverse_activation = [{
        **compressed_activation,
        "preactivation_lower": reverse.preactivation_lower.detach().cpu(),
        "preactivation_upper": reverse.preactivation_upper.detach().cpu(),
        "rho1": reverse.derivative_approximation_radii.detach().cpu(),
        "kind1": np.where(degrees == 2, "quadratic", "affine"),
        "affine_rho1": reverse.affine_derivative_approximation_radii.detach().cpu(),
        "quadratic_core_box_radius": torch.zeros_like(reverse.derivative_approximation_radii).cpu(),
        "relative_slope": reverse.derivative_relative_slopes.detach().cpu(),
    }]
    results[method_id] = {
        "method_id": method_id,
        "run_id": "pinn100d_shallow300_medium_hybrid_pz_uncompressed_reverse_symbolic",
        "Y": reverse.final.Y.interval_enclosure(),
        "J": reverse.final.J.interval_enclosure(),
        "Y_pz": reverse.final.Y,
        "J_pz": reverse.final.J,
        "activation": reverse_activation,
        "integrated_L2_pz": pz_l2_integrated,
        "integrated_W12_pz": pz_w12_integrated,
        "norms": {
            "L2_sq": _nonnegative_squared_interval(pz_l2_sq),
            "L2": norm_interval(pz_l2_sq),
            "W12_sq": _nonnegative_squared_interval(pz_w12_sq),
            "W12": norm_interval(pz_w12_sq),
        },
        "timings": {
            **reverse.timings,
            "onejet_construction_measured": reverse_forward_s,
            "L2_symbolic_integration": pz_l2_s,
            "W12_symbolic_integration": pz_w12_s,
            "W12_total": reverse_forward_s + pz_w12_s,
        },
    }
    return results


medium_results = _run_medium_methods(model)


In [7]:
def _interval_stats(lower, upper, scale):
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    midpoint = 0.5 * (lower + upper)
    radius = 0.5 * (upper - lower)
    width = upper - lower
    magnitude = np.maximum(np.abs(lower), np.abs(upper))
    mignitude = np.where((lower <= 0.0) & (upper >= 0.0), 0.0, np.minimum(np.abs(lower), np.abs(upper)))
    local = np.divide(radius, magnitude, out=np.zeros_like(radius), where=magnitude > 0.0)
    global_radius = np.divide(radius, scale, out=np.zeros_like(radius), where=scale > 0.0)
    return midpoint, radius, width, magnitude, mignitude, local, global_radius


def _family_metrics(base, method, quantity, lower, upper, derivative_order):
    lower = np.asarray(lower, dtype=float).reshape(-1)
    upper = np.asarray(upper, dtype=float).reshape(-1)
    scale = float(np.max(np.maximum(np.abs(lower), np.abs(upper)))) if lower.size else 0.0
    _, _, width, _, _, _, global_radius = _interval_stats(lower, upper, scale)
    specifications = [
        ("mean_width", "mean_weighted", float(np.mean(width))),
        ("max_width", "max", float(np.max(width))),
        ("q50_width", "q50", float(np.quantile(width, 0.50, method="linear"))),
        ("q90_width", "q90", float(np.quantile(width, 0.90, method="linear"))),
        ("q99_width", "q99", float(np.quantile(width, 0.99, method="linear"))),
        ("mean_global_normalized_radius", "mean_weighted", float(np.mean(global_radius))),
        ("max_global_normalized_radius", "max", float(np.max(global_radius))),
    ]
    rows = []
    for metric, aggregation, value in specifications:
        rows.append({**base, "quantity": quantity, "metric": metric,
                     "aggregation": aggregation, "derivative_order": derivative_order,
                     "value": value, "unit": "dimensionless", "status": "ok"})
    if quantity == "J":
        matrix_width = np.asarray(upper - lower, dtype=float)
        frobenius = float(np.linalg.norm(matrix_width.reshape(-1)))
        rows.extend([
            {**base, "quantity": "J", "metric": "mean_frobenius_width",
             "aggregation": "mean_weighted", "derivative_order": 1,
             "value": frobenius, "unit": "dimensionless", "status": "ok"},
            {**base, "quantity": "J", "metric": "max_frobenius_width",
             "aggregation": "max", "derivative_order": 1,
             "value": frobenius, "unit": "dimensionless", "status": "ok"},
        ])
    return rows


def _base_metric(result):
    return {
        "schema_version": SCHEMA_VERSION, "benchmark_level": BENCHMARK_LEVEL,
        "problem_id": PROBLEM_ID, "model_id": MODEL_ID,
        "method_id": result["method_id"], "run_id": result["run_id"],
        "split_id": SPLIT_ID, "layer": pd.NA, "neuron": pd.NA,
        "output_index": pd.NA, "input_index_a": pd.NA,
        "input_index_b": pd.NA, "cell_id": pd.NA,
    }


def _cell_interval_table(result):
    rows = []
    for quantity, enclosure in (("Y", result["Y"]), ("J", result["J"])):
        lower = np.asarray(_tensor(enclosure.lower), dtype=float)
        upper = np.asarray(_tensor(enclosure.upper), dtype=float)
        scale = float(np.max(np.maximum(np.abs(lower), np.abs(upper)))) if lower.size else 0.0
        midpoint, radius, width, magnitude, mignitude, local, global_radius = _interval_stats(lower, upper, scale)
        for index in np.ndindex(lower.shape):
            if quantity == "Y":
                output_index, input_a = (index[0] if index else 0), pd.NA
            else:
                output_index, input_a = index
            rows.append({
                "run_id": result["run_id"], "split_id": SPLIT_ID, "cell_id": 0,
                "cell_weight": 1.0, "quantity": quantity,
                "output_index": output_index, "input_index_a": input_a,
                "input_index_b": pd.NA, "lower": float(lower[index]),
                "upper": float(upper[index]), "midpoint": float(midpoint[index]),
                "radius": float(radius[index]), "width": float(width[index]),
                "magnitude": float(magnitude[index]), "mignitude": float(mignitude[index]),
                "local_relative_radius": float(local[index]),
                "global_normalized_radius": float(global_radius[index]),
                "sign_certified": int(not (lower[index] <= 0.0 <= upper[index])),
                "status": "ok",
                "local_relative_width": float(2.0 * local[index]),
            })
    return pd.DataFrame(rows, columns=CELL_INTERVAL_COLUMNS).sort_values(
        ["cell_id", "quantity", "output_index", "input_index_a", "input_index_b"],
        na_position="last", kind="stable", ignore_index=True,
    )


def _norm_table(result):
    rows = []
    for norm in ("L2", "W12"):
        for squared in (1, 0):
            key = norm + ("_sq" if squared else "")
            lower, upper = map(float, result["norms"][key])
            width = upper - lower
            volume_scale = VOLUME if squared else SQRT_VOLUME
            rows.append({
                "run_id": result["run_id"], "norm": norm, "squared": squared,
                "lower": lower, "upper": upper, "width": width,
                "relative_width": width / upper if upper > 0.0 else 0.0,
                "value_contribution_upper": pd.NA,
                "gradient_contribution_upper": pd.NA,
                "hessian_contribution_upper": pd.NA,
                "value_contribution_width": pd.NA,
                "gradient_contribution_width": pd.NA,
                "hessian_contribution_width": pd.NA,
                "status": "ok",
                "domain_volume": VOLUME,
                "domain_volume_normalized_lower": lower / volume_scale,
                "domain_volume_normalized_upper": upper / volume_scale,
                "domain_volume_normalized_width": width / volume_scale,
            })
    return pd.DataFrame(rows, columns=NORM_COLUMNS)


def _tanh_prime_hull(lower, upper):
    t_lo = np.tanh(lower)
    t_hi = np.tanh(upper)
    endpoint_lo = 1.0 - t_lo * t_lo
    endpoint_hi = 1.0 - t_hi * t_hi
    hull_lower = np.minimum(endpoint_lo, endpoint_hi)
    hull_upper = np.where((lower <= 0.0) & (upper >= 0.0), 1.0, np.maximum(endpoint_lo, endpoint_hi))
    return hull_lower, hull_upper


def _activation_tables(result):
    rows = []
    wide = {"neuron": np.arange(max(len(record["preactivation_lower"]) for record in result["activation"]))}
    for record in result["activation"]:
        layer = int(record["layer"])
        lower = np.asarray(record["preactivation_lower"], dtype=float)
        upper = np.asarray(record["preactivation_upper"], dtype=float)
        midpoint = 0.5 * (lower + upper)
        radius = 0.5 * (upper - lower)
        width = upper - lower
        y_lower = np.tanh(lower)
        y_upper = np.tanh(upper)
        d_lower, d_upper = _tanh_prime_hull(lower, upper)
        hulls = {0: (y_lower, y_upper), 1: (d_lower, d_upper)}
        post_lower = np.asarray(record["postactivation_lower"], dtype=float)
        post_upper = np.asarray(record["postactivation_upper"], dtype=float)
        y_scale = float(np.max(np.maximum(np.abs(post_lower), np.abs(post_upper))))
        y_normalized = (0.5 * (post_upper - post_lower) / y_scale) if y_scale > 0.0 else np.zeros_like(post_lower)
        padded = np.full(len(wide["neuron"]), np.nan)
        padded[:len(y_normalized)] = y_normalized
        wide[f"layer_{layer}"] = padded
        for derivative_order in (0, 1):
            hull_lower, hull_upper = hulls[derivative_order]
            scale = float(np.max(np.maximum(np.abs(hull_lower), np.abs(hull_upper))))
            if result["method_id"] == "interval":
                rho = 0.5 * (hull_upper - hull_lower)
                kinds = np.full(len(lower), "interval", dtype=object)
                noise_ids = [pd.NA] * len(lower)
            else:
                rho = np.asarray(record[f"rho{derivative_order}"], dtype=float)
                kinds = np.asarray(record.get(f"kind{derivative_order}", np.full(len(lower), "affine")), dtype=object)
                noise_ids = [f"eta_l{layer}_n{neuron}_r{derivative_order}" for neuron in range(len(lower))]
            normalized = rho / scale if scale > 0.0 else np.zeros_like(rho)
            for neuron in range(len(lower)):
                rows.append({
                    "run_id": result["run_id"], "split_id": SPLIT_ID,
                    "cell_id": 0, "cell_weight": 1.0, "layer": layer,
                    "neuron": neuron, "derivative_order": derivative_order,
                    "preactivation_lower": lower[neuron],
                    "preactivation_upper": upper[neuron],
                    "preactivation_midpoint": midpoint[neuron],
                    "preactivation_radius": radius[neuron],
                    "preactivation_width": width[neuron],
                    "approximation_kind": kinds[neuron],
                    "approximation_error_radius": rho[neuron],
                    "approximation_error_diameter": 2.0 * rho[neuron],
                    "activation_scale": scale,
                    "normalized_approximation_radius": normalized[neuron],
                    "noise_symbol_id": noise_ids[neuron],
                    "shared_noise_group": pd.NA, "status": "ok",
                })
    activation = pd.DataFrame(rows, columns=ACTIVATION_COLUMNS).sort_values(
        ["cell_id", "layer", "neuron", "derivative_order"], kind="stable", ignore_index=True,
    )
    wide_table = pd.DataFrame(wide)[["neuron"] + sorted([key for key in wide if key.startswith("layer_")])]
    return activation, wide_table


def _pz_complexity_row(result, quantity, pz):
    kinds = tuple(pz.noise_kinds)
    domain = {index for index, kind in enumerate(kinds) if kind == "domain"}
    approximation = set(range(len(kinds))) - domain
    support = list(pz.terms)
    mixed = sum(
        int(any(exp[index] for index in domain) and any(exp[index] for index in approximation))
        for exp in support
    )
    coefficient_dimension = int(np.prod(pz.shape)) if pz.shape else 1
    return {
        "run_id": result["run_id"], "quantity": quantity,
        "n_alpha": len(domain), "n_eta": len(approximation),
        "n_monomials": len(support), "n_mixed_monomials": mixed,
        "max_degree": max((sum(exp) for exp in support), default=0),
        "n_coefficients": coefficient_dimension * len(support), "status": "ok",
    }


def _complexity_table(result):
    if "Y_pz" in result:
        rows = [_pz_complexity_row(result, "Y", result["Y_pz"]),
                _pz_complexity_row(result, "J", result["J_pz"])]
    else:
        rows = [{"run_id": result["run_id"], "quantity": quantity, "status": "not_implemented"}
                for quantity in ("Y", "J")]
    rows.append({"run_id": result["run_id"], "quantity": "H", "status": "not_implemented"})
    return pd.DataFrame(rows).reindex(columns=COMPLEXITY_COLUMNS)


def _soundness_table(result):
    rows = []
    exact_values = {
        "Y": prediction.detach().cpu().reshape(-1, 1).numpy(),
        "J": network_jacobian.detach().cpu().reshape(-1, 1, DIM).numpy(),
    }
    for quantity, enclosure in (("Y", result["Y"]), ("J", result["J"])):
        lower = np.asarray(_tensor(enclosure.lower), dtype=float)
        upper = np.asarray(_tensor(enclosure.upper), dtype=float)
        values = exact_values[quantity]
        violation = np.maximum(np.maximum(lower - values, values - upper), 0.0)
        endpoints = np.concatenate([lower.reshape(-1), upper.reshape(-1)])
        rows.append({
            "run_id": result["run_id"], "quantity": quantity,
            "sample_count": values.shape[0],
            "failure_count": int(np.count_nonzero(np.any(violation > 0.0, axis=tuple(range(1, violation.ndim))))),
            "max_violation": float(np.max(violation)),
            "invalid_interval_count": int(np.count_nonzero(lower > upper)),
            "nan_endpoint_count": int(np.count_nonzero(np.isnan(endpoints))),
            "infinite_endpoint_count": int(np.count_nonzero(np.isinf(endpoints))),
            "status": "ok",
        })
    return pd.DataFrame(rows, columns=SOUNDNESS_COLUMNS)


def _metrics_table(result, cell_intervals, norms, complexity, timings, soundness):
    base = _base_metric(result)
    rows = []
    for quantity, derivative_order in (("Y", 0), ("J", 1)):
        family = cell_intervals[cell_intervals.quantity == quantity]
        rows.extend(_family_metrics(base, result["method_id"], quantity,
                                    family.lower, family.upper, derivative_order))
    for metric in ("mean_width", "max_width", "q50_width", "q90_width", "q99_width",
                   "mean_global_normalized_radius", "max_global_normalized_radius"):
        rows.append({**base, "quantity": "H", "metric": metric, "aggregation": "none",
                     "derivative_order": 2, "value": pd.NA, "unit": pd.NA,
                     "status": "not_implemented"})
    for _, row in norms.iterrows():
        quantity = row["norm"] + ("_sq" if row["squared"] else "")
        for metric in ("lower", "upper", "width", "relative_norm_width",
                       "domain_volume_normalized_lower",
                       "domain_volume_normalized_upper",
                       "domain_volume_normalized_width"):
            value = row["relative_width"] if metric == "relative_norm_width" else row[metric]
            rows.append({**base, "quantity": quantity, "metric": metric,
                         "aggregation": "none", "derivative_order": pd.NA,
                         "value": value, "unit": "dimensionless", "status": row["status"]})
    for residual_quantity, derivative_order in (("PDE_residual", 2), ("boundary_residual", 0), ("initial_residual", 0)):
        rows.append({**base, "quantity": residual_quantity, "metric": "linf_upper",
                     "aggregation": "max", "derivative_order": derivative_order, "value": pd.NA,
                     "unit": "dimensionless", "status": "not_implemented"})
    for _, row in complexity.iterrows():
        for column, metric in (("n_alpha", "n_alpha"), ("n_eta", "n_eta"),
                               ("n_monomials", "n_monomials"),
                               ("n_mixed_monomials", "n_mixed_monomials"),
                               ("max_degree", "max_degree")):
            rows.append({**base, "quantity": "complexity", "metric": metric,
                         "aggregation": "none", "derivative_order": pd.NA,
                         "value": row[column], "unit": "dimensionless", "status": row["status"],
                         "layer": pd.NA, "neuron": pd.NA})
    for _, row in timings.iterrows():
        rows.append({**base, "quantity": "runtime", "metric": "seconds",
                     "aggregation": row["stage"], "derivative_order": pd.NA,
                     "value": row["seconds"], "unit": "seconds", "status": row["status"]})
    rows.append({**base, "quantity": "memory", "metric": "bytes",
                 "aggregation": "peak", "derivative_order": pd.NA,
                 "value": pd.NA, "unit": "bytes", "status": "not_implemented"})
    for _, row in soundness.iterrows():
        for column, metric in (("failure_count", "failure_count"), ("max_violation", "max_violation")):
            rows.append({**base, "quantity": "soundness", "metric": metric,
                         "aggregation": row["quantity"], "derivative_order": pd.NA,
                         "value": row[column], "unit": "dimensionless", "status": row["status"]})
    return pd.DataFrame(rows).reindex(columns=METRICS_COLUMNS)

In [8]:
medium_outputs = {}
try:
    git_commit = subprocess.check_output(
        ["git", "rev-parse", "HEAD"], cwd=repo_root, text=True
    ).strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = None

for method_id, result in medium_results.items():
    method_dir = OUTPUT_ROOT / method_id
    method_dir.mkdir(parents=True, exist_ok=True)
    cell_intervals = _cell_interval_table(result)
    norms = _norm_table(result)
    activation, layer_radius_y = _activation_tables(result)
    complexity = _complexity_table(result)
    timings = pd.DataFrame([
        {"run_id": result["run_id"], "stage": stage, "seconds": seconds, "status": "ok"}
        for stage, seconds in result["timings"].items()
    ], columns=TIMING_COLUMNS)
    soundness = _soundness_table(result)
    metrics = _metrics_table(result, cell_intervals, norms, complexity, timings, soundness)

    assert list(metrics.columns) == METRICS_COLUMNS
    assert list(cell_intervals.columns) == CELL_INTERVAL_COLUMNS
    assert list(norms.columns) == NORM_COLUMNS
    assert list(activation.columns) == ACTIVATION_COLUMNS
    assert list(layer_radius_y.columns) == ["neuron", "layer_0"]
    assert list(complexity.columns) == COMPLEXITY_COLUMNS
    assert list(timings.columns) == TIMING_COLUMNS
    assert list(soundness.columns) == SOUNDNESS_COLUMNS
    assert activation.equals(activation.sort_values(
        ["cell_id", "layer", "neuron", "derivative_order"], kind="stable", ignore_index=True
    ))
    assert np.isclose(activation.cell_weight.groupby([activation.cell_id]).first().sum(), 1.0)
    assert not bool((norms.relative_width < 0.0).any() or (norms.relative_width > 1.0).any())
    assert int(soundness.failure_count.sum()) == 0
    assert np.allclose(
        cell_intervals.local_relative_width,
        2.0 * cell_intervals.local_relative_radius,
    )
    assert list(zip(norms["norm"], norms["squared"])) == [
        ("L2", 1), ("L2", 0), ("W12", 1), ("W12", 0),
    ]
    assert not norms[[
        "domain_volume", "domain_volume_normalized_lower",
        "domain_volume_normalized_upper", "domain_volume_normalized_width",
    ]].isna().any().any()
    norm_scales = np.where(norms.squared.astype(bool), VOLUME, SQRT_VOLUME)
    assert np.all(norms.domain_volume.to_numpy() == VOLUME)
    assert np.allclose(norms.domain_volume_normalized_lower, norms.lower / norm_scales)
    assert np.allclose(norms.domain_volume_normalized_upper, norms.upper / norm_scales)
    assert np.allclose(norms.domain_volume_normalized_width, norms.width / norm_scales)

    metadata = {
        "schema_version": SCHEMA_VERSION,
        "benchmark_level": BENCHMARK_LEVEL,
        "problem_id": PROBLEM_ID,
        "model_id": MODEL_ID,
        "method_id": method_id,
        "git_commit": git_commit,
        "dtype": str(torch.get_default_dtype()).replace("torch.", ""),
        "device": "cpu",
        "quantile_interpolation": "linear",
        "cell_average": "volume_weighted",
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }
    (method_dir / "benchmark_metadata.json").write_text(
        json.dumps(metadata, indent=2) + "\n", encoding="utf-8"
    )
    for filename, frame in {
        "metrics.csv": metrics,
        "cell_intervals.csv": cell_intervals,
        "norms.csv": norms,
        "complexity.csv": complexity,
        "timings.csv": timings,
        "soundness.csv": soundness,
        "activation_approximation.csv": activation,
        "layer_normalized_radius_Y.csv": layer_radius_y,
    }.items():
        frame.to_csv(method_dir / filename, index=False, na_rep="NA")
    medium_outputs[method_id] = {
        "metadata": metadata, "metrics": metrics, "cell_intervals": cell_intervals,
        "norms": norms, "complexity": complexity, "timings": timings,
        "soundness": soundness, "activation": activation,
        "layer_radius_Y": layer_radius_y,
    }

medium_norms_summary = pd.concat([
    output["norms"].assign(method_id=method_id)
    for method_id, output in medium_outputs.items()
], ignore_index=True)[[
    "method_id", "norm", "squared", "lower", "upper", "width", "relative_width",
    "domain_volume", "domain_volume_normalized_lower",
    "domain_volume_normalized_upper", "domain_volume_normalized_width", "status"
]]
medium_norms_summary

,method_id,norm,squared,lower,upper,width,relative_width,domain_volume,domain_volume_normalized_lower,domain_volume_normalized_upper,domain_volume_normalized_width,status
0,interval,L2,1,0.0,3.188219e-68,3.188219e-68,1.0,1.267651e-70,0.0,251.506108,251.506108,ok
1,interval,L2,0,0.0,1.785558e-34,1.785558e-34,1.0,1.267651e-70,0.0,15.858944,15.858944,ok
2,interval,W12,1,0.0,4.013416e-68,4.013416e-68,1.0,1.267651e-70,0.0,316.602733,316.602733,ok
3,interval,W12,0,0.0,2.003351e-34,2.003351e-34,1.0,1.267651e-70,0.0,17.793334,17.793334,ok
4,affine_pz_topk96_symbolic,L2,1,0.0,1.244074e-69,1.244074e-69,1.0,1.267651e-70,0.0,9.814011,9.814011,ok
5,affine_pz_topk96_symbolic,L2,0,0.0,3.527143e-35,3.527143e-35,1.0,1.267651e-70,0.0,3.132732,3.132732,ok
6,affine_pz_topk96_symbolic,W12,1,0.0,9.340887e-69,9.340887e-69,1.0,1.267651e-70,0.0,73.686608,73.686608,ok
7,affine_pz_topk96_symbolic,W12,0,0.0,9.664827e-35,9.664827e-35,1.0,1.267651e-70,0.0,8.584090,8.584090,ok
8,hybrid_pz_topk96_B_symbolic,L2,1,0.0,1.244074e-69,1.244074e-69,1.0,1.267651e-70,0.0,9.814011,9.814011,ok
9,hybrid_pz_topk96_B_symbolic,L2,0,0.0,3.527143e-35,3.527143e-35,1.0,1.267651e-70,0.0,3.132732,3.132732,ok


### Canonical squared and unsquared norm intervals

In [9]:
normalized_norm_view = medium_norms_summary[medium_norms_summary.squared == 0].copy()
normalized_norm_view[[
    "method_id", "norm", "domain_volume_normalized_lower",
    "domain_volume_normalized_upper", "domain_volume_normalized_width",
    "relative_width",
]]

,method_id,norm,domain_volume_normalized_lower,domain_volume_normalized_upper,domain_volume_normalized_width,relative_width
1,interval,L2,0.0,15.858944,15.858944,1.0
3,interval,W12,0.0,17.793334,17.793334,1.0
5,affine_pz_topk96_symbolic,L2,0.0,3.132732,3.132732,1.0
7,affine_pz_topk96_symbolic,W12,0.0,8.584090,8.584090,1.0
9,hybrid_pz_topk96_B_symbolic,L2,0.0,3.132732,3.132732,1.0
11,hybrid_pz_topk96_B_symbolic,W12,0.0,8.779882,8.779882,1.0
13,hybrid_pz_uncompressed_reverse_symbolic,L2,0.0,3.132732,3.132732,1.0
15,hybrid_pz_uncompressed_reverse_symbolic,W12,0.0,4.769351,4.769351,1.0


### Final enclosure, complexity, and runtime diagnostics

In [10]:
medium_enclosure_summary = pd.concat([
    output["metrics"].query(
        "quantity in ['Y', 'J'] and metric in ['mean_width', 'max_width', "
        "'mean_global_normalized_radius', 'max_global_normalized_radius', "
        "'mean_frobenius_width', 'max_frobenius_width']"
    )[["method_id", "quantity", "metric", "aggregation", "value", "status"]]
    for output in medium_outputs.values()
], ignore_index=True)
medium_enclosure_summary

,method_id,quantity,metric,aggregation,value,status
0,interval,Y,mean_width,mean_weighted,31.450925,ok
1,interval,Y,max_width,max,31.450925,ok
2,interval,Y,mean_global_normalized_radius,mean_weighted,0.991583,ok
3,interval,Y,max_global_normalized_radius,max,0.991583,ok
4,interval,J,mean_width,mean_weighted,1.353524,ok
5,interval,J,max_width,max,1.682731,ok
6,interval,J,mean_global_normalized_radius,mean_weighted,0.694703,ok
7,interval,J,max_global_normalized_radius,max,0.863669,ok
8,interval,J,mean_frobenius_width,mean_weighted,13.575215,ok
9,interval,J,max_frobenius_width,max,13.575215,ok


In [11]:
pd.concat([
    output["complexity"].assign(method_id=method_id)
    for method_id, output in medium_outputs.items()
], ignore_index=True)[["method_id"] + COMPLEXITY_COLUMNS]

,method_id,run_id,quantity,n_alpha,n_eta,n_monomials,n_mixed_monomials,max_degree,n_coefficients,status
0,interval,pinn100d_shallow300_medium_interval,Y,NaN,NaN,NaN,NaN,NaN,NaN,not_implemented
1,interval,pinn100d_shallow300_medium_interval,J,NaN,NaN,NaN,NaN,NaN,NaN,not_implemented
2,interval,pinn100d_shallow300_medium_interval,H,NaN,NaN,NaN,NaN,NaN,NaN,not_implemented
3,affine_pz_topk96_symbolic,pinn100d_shallow300_medium_affine_pz_topk96_sy...,Y,100.0,400.0,400.0,0.0,1.0,400.0,ok
4,affine_pz_topk96_symbolic,pinn100d_shallow300_medium_affine_pz_topk96_sy...,J,100.0,400.0,196.0,0.0,1.0,19600.0,ok
5,affine_pz_topk96_symbolic,pinn100d_shallow300_medium_affine_pz_topk96_sy...,H,NaN,NaN,NaN,NaN,NaN,NaN,not_implemented
6,hybrid_pz_topk96_B_symbolic,pinn100d_shallow300_medium_hybrid_pz_topk96_B_...,Y,100.0,400.0,400.0,0.0,1.0,400.0,ok
7,hybrid_pz_topk96_B_symbolic,pinn100d_shallow300_medium_hybrid_pz_topk96_B_...,J,100.0,400.0,196.0,0.0,1.0,19600.0,ok
8,hybrid_pz_topk96_B_symbolic,pinn100d_shallow300_medium_hybrid_pz_topk96_B_...,H,NaN,NaN,NaN,NaN,NaN,NaN,not_implemented
9,hybrid_pz_uncompressed_reverse_symbolic,pinn100d_shallow300_medium_hybrid_pz_uncompres...,Y,100.0,600.0,400.0,0.0,1.0,400.0,ok


### Per-neuron activation diagnostics

In [12]:
medium_activation_summary = pd.concat([
    output["activation"].assign(method_id=method_id)
    for method_id, output in medium_outputs.items()
], ignore_index=True)
medium_activation_layer_summary = (
    medium_activation_summary
    .groupby(["method_id", "layer", "derivative_order"], sort=True)
    .agg(
        preactivation_radius_mean=("preactivation_radius", "mean"),
        preactivation_radius_max=("preactivation_radius", "max"),
        approximation_error_radius_mean=("approximation_error_radius", "mean"),
        approximation_error_radius_max=("approximation_error_radius", "max"),
        normalized_approximation_radius_mean=("normalized_approximation_radius", "mean"),
        normalized_approximation_radius_max=("normalized_approximation_radius", "max"),
    )
    .reset_index()
)
medium_activation_layer_summary

,method_id,layer,derivative_order,preactivation_radius_mean,preactivation_radius_max,approximation_error_radius_mean,approximation_error_radius_max,normalized_approximation_radius_mean,normalized_approximation_radius_max
0,affine_pz_topk96_symbolic,0,0,1.000772,1.671261,0.084834,0.218043,0.091023,0.233953
1,affine_pz_topk96_symbolic,0,1,1.000772,1.671261,0.283051,0.434049,0.283051,0.434049
2,hybrid_pz_topk96_B_symbolic,0,0,1.000772,1.671261,0.084834,0.218043,0.091023,0.233953
3,hybrid_pz_topk96_B_symbolic,0,1,1.000772,1.671261,0.279491,0.414173,0.279491,0.414173
4,hybrid_pz_uncompressed_reverse_symbolic,0,0,1.000772,1.671261,0.084834,0.218043,0.091023,0.233953
5,hybrid_pz_uncompressed_reverse_symbolic,0,1,1.000772,1.671261,0.279491,0.414173,0.279491,0.414173
6,interval,0,0,1.000772,1.671261,0.747199,0.931718,0.801719,0.999701
7,interval,0,1,1.000772,1.671261,0.288093,0.434309,0.288093,0.434309


In [13]:
activation_views = {}
for method_id, output in medium_outputs.items():
    for derivative_order in (0, 1):
        view = output["activation"].query("derivative_order == @derivative_order")[
            ["layer", "neuron", "preactivation_lower", "preactivation_upper",
             "approximation_kind", "approximation_error_radius",
             "normalized_approximation_radius", "status"]
        ].reset_index(drop=True)
        activation_views[(method_id, derivative_order)] = view
        display(method_id, f"derivative_order={derivative_order}", view)

'interval'

'derivative_order=0'

,layer,neuron,preactivation_lower,preactivation_upper,approximation_kind,approximation_error_radius,normalized_approximation_radius,status
0,0,0,-0.800089,0.790877,interval,0.661496,0.709762,ok
1,0,1,-0.666810,0.693385,interval,0.591515,0.634675,ok
2,0,2,-0.840969,0.870502,interval,0.693975,0.744612,ok
3,0,3,-0.813165,0.846186,interval,0.680202,0.729833,ok
4,0,4,-1.037626,1.001014,interval,0.769484,0.825630,ok
...,...,...,...,...,...,...,...,...
295,0,295,-1.416549,1.448714,interval,0.892158,0.957254,ok
296,0,296,-0.736982,0.710152,interval,0.619045,0.664214,ok
297,0,297,-0.982514,1.013824,interval,0.760746,0.816254,ok
298,0,298,-0.833206,0.854856,interval,0.687896,0.738088,ok


'interval'

'derivative_order=1'

...

'derivative_order=0'

,layer,neuron,preactivation_lower,preactivation_upper,approximation_kind,approximation_error_radius,normalized_approximation_radius,status
0,0,0,-0.800089,0.790877,affine,0.047761,0.051246,ok
1,0,1,-0.666810,0.693385,affine,0.032204,0.034554,ok
2,0,2,-0.840969,0.870502,affine,0.057050,0.061212,ok
3,0,3,-0.813165,0.846186,affine,0.052974,0.056839,ok
4,0,4,-1.037626,1.001014,affine,0.085345,0.091572,ok
...,...,...,...,...,...,...,...,...
295,0,295,-1.416549,1.448714,affine,0.168543,0.180841,ok
296,0,296,-0.736982,0.710152,affine,0.037730,0.040483,ok
297,0,297,-0.982514,1.013824,affine,0.081450,0.087393,ok
298,0,298,-0.833206,0.854856,affine,0.055178,0.059205,ok


'hybrid_pz_uncompressed_reverse_symbolic'

'derivative_order=1'

,layer,neuron,preactivation_lower,preactivation_upper,approximation_kind,approximation_error_radius,normalized_approximation_radius,status
0,0,0,-0.800089,0.790877,affine,0.218784,0.218784,ok
1,0,1,-0.666810,0.693385,affine,0.174911,0.174911,ok
2,0,2,-0.840969,0.870502,affine,0.240758,0.240758,ok
3,0,3,-0.813165,0.846186,affine,0.231283,0.231283,ok
4,0,4,-1.037626,1.001014,affine,0.295994,0.295994,ok
...,...,...,...,...,...,...,...,...
295,0,295,-1.416549,1.448714,affine,0.397947,0.397947,ok
296,0,296,-0.736982,0.710152,affine,0.191573,0.191573,ok
297,0,297,-0.982514,1.013824,affine,0.289323,0.289323,ok
298,0,298,-0.833206,0.854856,affine,0.236577,0.236577,ok


### Hybrid switch and quadratic-core diagnostics

In [14]:

hybrid_record = medium_results["hybrid_pz_uncompressed_reverse_symbolic"]["activation"][0]
hybrid_neuron_diagnostics = pd.DataFrame({
    "neuron": np.arange(len(hybrid_record["rho1"])),
    "preactivation_lower": np.asarray(hybrid_record["preactivation_lower"], dtype=float),
    "preactivation_upper": np.asarray(hybrid_record["preactivation_upper"], dtype=float),
    "relative_slope": np.asarray(hybrid_record["relative_slope"], dtype=float),
    "approximation_kind": np.asarray(hybrid_record["kind1"], dtype=object),
    "affine_approximation_error_radius": np.asarray(hybrid_record["affine_rho1"], dtype=float),
    "selected_approximation_error_radius": np.asarray(hybrid_record["rho1"], dtype=float),
    "quadratic_core_box_radius": np.asarray(hybrid_record["quadratic_core_box_radius"], dtype=float),
})
hybrid_summary = pd.DataFrame([{
    "flatness_threshold": 0.01,
    "zero_crossing_neurons": int(np.count_nonzero(
        (hybrid_neuron_diagnostics.preactivation_lower <= 0.0)
        & (hybrid_neuron_diagnostics.preactivation_upper >= 0.0)
    )),
    "quadratic_neurons": int(np.count_nonzero(hybrid_neuron_diagnostics.approximation_kind == "quadratic")),
    "mean_affine_rho1": hybrid_neuron_diagnostics.affine_approximation_error_radius.mean(),
    "mean_selected_rho1": hybrid_neuron_diagnostics.selected_approximation_error_radius.mean(),
    "mean_quadratic_core_box_radius": hybrid_neuron_diagnostics.quadratic_core_box_radius.mean(),
}])
display(hybrid_summary)
display(hybrid_neuron_diagnostics.query("approximation_kind == 'quadratic'").reset_index(drop=True))


,flatness_threshold,zero_crossing_neurons,quadratic_neurons,mean_affine_rho1,mean_selected_rho1,mean_quadratic_core_box_radius
0,0.01,300,4,0.283051,0.279491,0.0


,neuron,preactivation_lower,preactivation_upper,relative_slope,approximation_kind,affine_approximation_error_radius,selected_approximation_error_radius,quadratic_core_box_radius
0,18,-1.673373,1.669149,0.001195,quadratic,0.434049,0.135497,0.0
1,56,-0.708344,0.703615,0.009753,quadratic,0.184920,0.014351,0.0
2,60,-1.586053,1.573384,0.004301,quadratic,0.421866,0.123005,0.0
3,101,-1.583788,1.580123,0.001241,quadratic,0.422194,0.122294,0.0


### Required layerwise normalized postactivation-radius tables

In [15]:
for method_id, output in medium_outputs.items():
    display(method_id, output["layer_radius_Y"])

'interval'

,neuron,layer_0
0,0,0.709762
1,1,0.634675
2,2,0.744612
3,3,0.729833
4,4,0.825630
...,...,...
295,295,0.957255
296,296,0.664214
297,297,0.816254
298,298,0.738089


'affine_pz_topk96_symbolic'

,neuron,layer_0
0,0,0.616242
1,1,0.541922
2,2,0.652533
3,3,0.637024
4,4,0.742724
...,...,...
295,295,0.921597
296,296,0.570643
297,297,0.731747
298,298,0.645625


'hybrid_pz_topk96_B_symbolic'

,neuron,layer_0
0,0,0.616242
1,1,0.541922
2,2,0.652533
3,3,0.637024
4,4,0.742724
...,...,...
295,295,0.921597
296,296,0.570643
297,297,0.731747
298,298,0.645625


'hybrid_pz_uncompressed_reverse_symbolic'

,neuron,layer_0
0,0,0.616242
1,1,0.541922
2,2,0.652533
3,3,0.637024
4,4,0.742724
...,...,...
295,295,0.921597
296,296,0.570643
297,297,0.731747
298,298,0.645625


### Ranked worst activation enclosures

In [16]:
largest_absolute_activation_radii = (
    medium_activation_summary
    .sort_values("approximation_error_radius", ascending=False, kind="stable")
    [["method_id", "layer", "neuron", "derivative_order", "preactivation_lower",
      "preactivation_upper", "approximation_kind", "approximation_error_radius",
      "normalized_approximation_radius"]]
    .head(20).reset_index(drop=True)
)
largest_normalized_activation_radii = (
    medium_activation_summary
    .sort_values("normalized_approximation_radius", ascending=False, kind="stable")
    [["method_id", "layer", "neuron", "derivative_order", "preactivation_lower",
      "preactivation_upper", "approximation_kind", "approximation_error_radius",
      "normalized_approximation_radius"]]
    .head(20).reset_index(drop=True)
)
largest_absolute_activation_radii, largest_normalized_activation_radii

(   method_id  ...  normalized_approximation_radius
 0   interval  ...                         0.999701
 1   interval  ...                         0.985955
 2   interval  ...                         0.985575
 3   interval  ...                         0.976566
 4   interval  ...                         0.975547
 5   interval  ...                         0.975511
 6   interval  ...                         0.966316
 7   interval  ...                         0.963651
 8   interval  ...                         0.963562
 9   interval  ...                         0.963072
 10  interval  ...                         0.961890
 11  interval  ...                         0.961794
 12  interval  ...                         0.957254
 13  interval  ...                         0.953726
 14  interval  ...                         0.953364
 15  interval  ...                         0.951152
 16  interval  ...                         0.950026
 17  interval  ...                         0.948308
 18  interva

### Validity and sampled-containment diagnostics

In [17]:
pd.concat([
    output["soundness"].assign(method_id=method_id)
    for method_id, output in medium_outputs.items()
], ignore_index=True)[["method_id"] + SOUNDNESS_COLUMNS]

,method_id,run_id,quantity,sample_count,failure_count,max_violation,invalid_interval_count,nan_endpoint_count,infinite_endpoint_count,status
0,interval,pinn100d_shallow300_medium_interval,Y,16384,0,0.0,0,0,0,ok
1,interval,pinn100d_shallow300_medium_interval,J,16384,0,0.0,0,0,0,ok
2,affine_pz_topk96_symbolic,pinn100d_shallow300_medium_affine_pz_topk96_sy...,Y,16384,0,0.0,0,0,0,ok
3,affine_pz_topk96_symbolic,pinn100d_shallow300_medium_affine_pz_topk96_sy...,J,16384,0,0.0,0,0,0,ok
4,hybrid_pz_topk96_B_symbolic,pinn100d_shallow300_medium_hybrid_pz_topk96_B_...,Y,16384,0,0.0,0,0,0,ok
5,hybrid_pz_topk96_B_symbolic,pinn100d_shallow300_medium_hybrid_pz_topk96_B_...,J,16384,0,0.0,0,0,0,ok
6,hybrid_pz_uncompressed_reverse_symbolic,pinn100d_shallow300_medium_hybrid_pz_uncompres...,Y,16384,0,0.0,0,0,0,ok
7,hybrid_pz_uncompressed_reverse_symbolic,pinn100d_shallow300_medium_hybrid_pz_uncompres...,J,16384,0,0.0,0,0,0,ok


## Interpretation

The shallow architecture is a useful positive result for certifiability: both
PZ variants are far tighter than interval arithmetic and their certified
$W^{1,2}$ upper bounds are much closer to the sampled network norm than in the
three-hidden-layer benchmark.  The strict hybrid switch selects only the most
symmetric derivative bumps.  Its local approximation-error radii decrease, but
with the current expanded Top-96 representation the resulting degree-two terms
still introduce enough reduction radius to make the final hybrid certificate
slightly wider than the purely affine derivative enclosure.  The notebook
therefore reports both quantities separately rather than attributing the final
width to the quadratic approximation itself.